# 01 — Load and Inspect a Replicate Cache

This notebook opens a pre-built replicate cache (`assets.npz`) and explores every field: the standardised feature matrix, VAE and PCA embeddings, evaluation splits, geographic metadata, and — crucially — the audit trail showing which target columns were removed to prevent leakage.

**Prerequisite:** run `python scripts/launchers/build_caches.py --reps 0` before opening this notebook.

**Runtime:** < 1 second.

**Functions demonstrated:**
- [`load_replicate_cache`](../docs/api/data.md#coreset_selectiondataload_replicate_cache)
- [`build_geo_info`](../docs/api/geo.md#coreset_selectiongeobuild_geo_info)
- [`validate_no_leakage`](../docs/api/data.md#coreset_selectiondatavalidate_no_leakage)

In [ ]:
# Standard imports + ensure repo root is on sys.path
import sys
from pathlib import Path

REPO = Path.cwd().resolve()
while REPO != REPO.parent and not (REPO / 'coreset_selection').is_dir():
    REPO = REPO.parent
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import matplotlib.pyplot as plt

print(f'Repo root: {REPO}')

## 1. Locate the cache

`build_caches.py` writes to `experiments/adaptive_tau_k100_ps_vae/replicate_cache/rep{id:02d}/assets.npz`.
If you built only `rep0`, that's the file we'll use.

In [ ]:
cache_path = REPO / 'experiments' / 'adaptive_tau_k100_ps_vae' / 'replicate_cache' / 'rep00' / 'assets.npz'
print('Exists:', cache_path.exists())
print('Path:  ', cache_path)

## 2. Load the cache

`load_replicate_cache` reads the `.npz` and returns a `ReplicateAssets` dataclass with structured fields.

In [ ]:
from coreset_selection.data import load_replicate_cache

assets = load_replicate_cache(str(cache_path))

print('X_scaled:       ', assets.X_scaled.shape)
print('Z_vae:          ', None if assets.Z_vae is None else assets.Z_vae.shape)
print('Z_pca:          ', None if assets.Z_pca is None else assets.Z_pca.shape)
print('eval_idx:       ', assets.eval_idx.shape)
print('eval_train_idx: ', assets.eval_train_idx.shape)
print('eval_test_idx:  ', assets.eval_test_idx.shape)
print('state_labels:   ', assets.state_labels.shape, '(unique:', len(np.unique(assets.state_labels)), ')')

## 3. Inspect the target-leakage audit trail

`build_replicate_cache` detects known downstream targets and removes them *before* VAE/PCA training. The removed column names are recorded in `assets.metadata['removed_target_columns']`.

In [ ]:
removed = assets.metadata.get('removed_target_columns', [])
print(f'{len(removed)} target columns removed before representation learning:')
for name in list(removed)[:20]:
    print(f'  - {name}')
if len(removed) > 20:
    print(f'  ... and {len(removed) - 20} more.')

## 4. Build `GeoInfo` and inspect group structure

In [ ]:
from coreset_selection.geo import build_geo_info

geo = build_geo_info(assets.state_labels, population_weights=assets.population)
print(f'G = {geo.G} groups, N = {geo.N} points\n')
print(f'{"State":<6} {"n_g":>6} {"pi (muni)":>12} {"pi_pop":>12}')
for g in range(min(10, geo.G)):
    pi_pop = geo.pi_pop[g] if geo.pi_pop is not None else float('nan')
    print(f'{geo.groups[g]:<6} {geo.group_sizes[g]:>6d} {geo.pi[g]:>12.4f} {pi_pop:>12.4f}')
if geo.G > 10:
    print(f'... ({geo.G - 10} more states)')

## 5. Visualise the population distribution across states

In [ ]:
order = np.argsort(-geo.pi_pop) if geo.pi_pop is not None else np.argsort(-geo.pi)
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(geo.G)
ax.bar(x, geo.pi[order], width=0.4, label='municipality-share π_g')
if geo.pi_pop is not None:
    ax.bar(x + 0.4, geo.pi_pop[order], width=0.4, label='population-share π_g^(pop)')
ax.set_xticks(x + 0.2)
ax.set_xticklabels([geo.groups[i] for i in order], rotation=45, ha='right')
ax.set_ylabel('share')
ax.set_title('Target distributions by state')
ax.legend()
plt.tight_layout()
plt.show()

## What's next

- [02_run_minimal_experiment.ipynb](./02_run_minimal_experiment.ipynb) — run a full NSGA-II experiment programmatically.
- [docs/api/data.md](../docs/api/data.md) — full API for every data-layer symbol.